# FLA-Lang: Code Validity Predictor (Naive Bayes ML Model)
### Course: Formal Languages and Automata (FLA)
**Project:** FLA-Lang Compiler Front-End Using Formal Languages and Automata  
**Objective:** Train a character n-gram Multinomial Naive Bayes classifier to statistically classify source code validity consistent with the FLA-Lang context-free grammar, and export inference parameters to `ml_model.json` for client-side execution.

---

## 1. Setup and Library Imports

In [ ]:
import json
import math
import random
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully.")

## 2. Dataset Generation Consistent with FLA-Lang Grammar
The dataset generates pairs `(code_snippet, label)` where `1 = valid` and `0 = invalid`.
- **Valid snippets** follow the FLA-Lang LL(1) grammar: declarations (`int`, `float`, `bool`), assignments, arithmetic expressions with precedence, `if-else` conditionals, and `while` loops.
- **Invalid snippets** contain syntax errors: missing semicolons, unmatched parentheses/braces, consecutive operators, identifiers starting with digits, malformed decimals, and illegal characters (`@`, `$`, `#`, `~`, `^`, `%`).

In [ ]:
def generate_fla_dataset():
    valid_samples = [
        # Declarations
        "int x;", "float y;", "bool flag;", "int a;", "float rate;", "bool active;",
        "int count = 0;", "int x = 10;", "float rate = 12.75;", "float temp = 98.6;",
        "bool flag = 1;", "bool isReady = 0;", "int num = 42;", "float pi = 3.14;",
        "int step = 1;", "int maxLimit = 100;", "float score = 0.0;",
        
        # Assignments & Arithmetic Expressions
        "x = a + 5;", "y = y * 2.0 + 1.5;", "result = (a + 5) * b - c / 2;",
        "count = count + 1;", "total = total + item;", "diff = a - b;", "prod = x * y;",
        "ratio = total / count;", "x = (a + b) * (c - d);", "y = -x + 10;",
        "val = (x / y) + (a * b);", "score = score + 10.5;", "x = a + b + c + d;",
        "res = a * b * c;", "p = (x + y + z) / 3;",

        # Multi-statement Programs
        "int x = 10;\nx = a + 5;",
        "float rate = 12.5;\nresult = (a + 5) * b - c / 2;",
        "int a = 1;\nint b = 2;\nint c = a + b;",
        "int x = 0;\nx = x + 1;\nx = x * 2;",
        "float x = 1.5;\nfloat y = 2.5;\nfloat z = (x + y) * 2.0;",
        "int count = 0;\nbool done = 0;\ncount = count + 1;",
        "int a = 10;\nint b = 20;\na = b;\nb = a;",

        # Conditionals
        "if (x > 10) {\n  y = 1;\n}",
        "if (x > 10) {\n  y = 1;\n} else {\n  y = 0;\n}",
        "if (flag == 1) {\n  count = count + 1;\n}",
        "if (a < b) {\n  min = a;\n} else {\n  min = b;\n}",
        "if (score > 50) {\n  pass = 1;\n} else {\n  pass = 0;\n}",

        # While Loops
        "while (count < 5) {\n  count = count + 1;\n}",
        "while (x > 0) {\n  x = x - 1;\n}",
        "while (i < 10) {\n  sum = sum + i;\n  i = i + 1;\n}",
        "while (total < 100) {\n  total = total + step;\n}",
        "while (running == 1) {\n  count = count + 1;\n}",

        # Blocks
        "int x = 0;\nwhile (x < 10) {\n  if (x > 5) {\n    y = 1;\n  } else {\n    y = 0;\n  }\n  x = x + 1;\n}",
        "{ int temp = 0; temp = temp + 1; }"
    ]

    invalid_samples = [
        # Missing Semicolons
        "int x = 10", "x = a + 5", "float rate = 12.5", "int x = 10\nx = 20;",
        "int a = 1\nint b = 2;", "count = count + 1", "bool flag = 1",

        # Mismatched Parentheses
        "x = (a + 5;", "x = a + 5);", "x = ((a + 5);", "if (x > 10 {\n  y = 1;\n}",
        "if x > 10) {\n  y = 1;\n}", "while (count < 5 {\n  count = count + 1;\n}",

        # Mismatched Braces
        "if (x > 0) {\n  y = 1;\n", "while (x < 5) {\n  x = x + 1;\n",
        "{ int x = 1; x = x + 2;", "if (x > 0) {\n  y = 1;\n} else {\n  y = 2;",

        # Illegal Characters
        "int x = 10 @ 5;", "float $money = 100.0;", "int x = 10 # comment;",
        "x = a & b;", "val = ~x;", "x = a ^ b;", "int x = 10 % 3;", "int _count = 0;",

        # Consecutive / Malformed Operators
        "x = a + * 5;", "y = 10 / / 2;", "x = = 10;", "x = a ++ 1;", "x = a / * b;",

        # Identifiers Starting with Digits
        "int 1x = 10;", "float 9total = 0.0;", "2a = 5;", "int 3count = 1;",

        # Malformed Numbers
        "float v = 3.14.15;", "int x = 10.;", "float y = .5;", "x = 12.34.56;",

        # Incomplete / Malformed Statements
        "x = ;", "int x = ;", "if () {\n  y = 1;\n}", "while () {\n  x = 1;\n}",
        "if {\n  y = 1;\n}", "else {\n  y = 2;\n}", "string s = hello;", "int if = 5;"
    ]

    # Systematic synthesized additions
    vars_list = ['x', 'y', 'z', 'a', 'b', 'c', 'sum', 'total', 'count', 'rate', 'item', 'score', 'num', 'step', 'limit']
    types_list = ['int', 'float', 'bool']
    nums_list = ['0', '1', '2', '5', '10', '25', '100', '3.14', '0.5', '12.5']
    ops_list = ['+', '-', '*', '/']
    comp_list = ['>', '<', '==']

    rng = random.Random(42)

    for _ in range(120):
        v = rng.choice(vars_list)
        t = rng.choice(types_list)
        n = rng.choice(nums_list) if t != 'bool' else rng.choice(['0', '1'])
        valid_samples.append(f"{t} {v};");
        valid_samples.append(f"{t} {v} = {n};");
        v1, v2 = rng.choice(vars_list), rng.choice(vars_list)
        op = rng.choice(ops_list)
        valid_samples.append(f"{v1} = {v2} {op} {n};");
        valid_samples.append(f"{v1} = ({v2} {op} {n}) * {rng.choice(nums_list)};");
        cmp_op = rng.choice(comp_list)
        valid_samples.append(f"if ({v1} {cmp_op} {n}) {{\n  {v2} = {v1} + 1;\n}}");
        valid_samples.append(f"while ({v1} {cmp_op} {n}) {{\n  {v1} = {v1} + 1;\n}}");
        valid_samples.append(f"{t} {v} = {n};\n{v1} = {v2} {op} {n};");
        valid_samples.append(f"{t} {v} = {n}; {v1} = {v2} {op} {n};");

    for _ in range(120):
        v = rng.choice(vars_list)
        t = rng.choice(types_list)
        n = rng.choice(nums_list)
        v1, v2 = rng.choice(vars_list), rng.choice(vars_list)
        op = rng.choice(ops_list)
        invalid_samples.append(f"{t} {v} = {n}");
        invalid_samples.append(f"{v1} = {v2} {op} {n}");
        invalid_samples.append(f"{t} {v} = {n}\n{v1} = {v2} {op} {n};");
        invalid_samples.append(f"{t} {v} = {n}; {v1} = {v2} {op} {n}");
        bad_char = rng.choice(['@', '$', '#', '&', '~', '^', '%', '!', '?'])
        invalid_samples.append(f"{t} {v} = {n} {bad_char} 5;");
        invalid_samples.append(f"{bad_char} {v} = {n};");
        invalid_samples.append(f"{v} = a {op} {op} {n};");
        invalid_samples.append(f"{t} {rng.randint(1, 9)}{v} = {n};");
        invalid_samples.append(f"{v1} = ({v2} {op} {n};");
        invalid_samples.append(f"{v1} = {v2} {op} {n});");
        invalid_samples.append(f"float {v} = {n}.{rng.randint(1, 9)}.{rng.randint(1, 9)};");

    # Balance dataset 1:1
    rng.shuffle(valid_samples)
    rng.shuffle(invalid_samples)
    min_len = min(len(valid_samples), len(invalid_samples))
    valid_samples = valid_samples[:min_len]
    invalid_samples = invalid_samples[:min_len]

    X = valid_samples + invalid_samples
    y = [1] * len(valid_samples) + [0] * len(invalid_samples)

    # Shuffle combined dataset
    combined = list(zip(X, y))
    rng.shuffle(combined)
    X_shuffled, y_shuffled = zip(*combined)
    return list(X_shuffled), list(y_shuffled)

X, y = generate_fla_dataset()
print(f"Dataset generated successfully: {len(X)} samples total (50% Valid, 50% Invalid)")

## 3. Train/Test Split and Model Training with scikit-learn
- Uses character n-grams (`ngram_range=(1, 4)`)
- Fits `CountVectorizer` and `MultinomialNB(alpha=1.0)`

In [ ]:
# 80% train, 20% test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# Character n-gram CountVectorizer
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 4), min_df=2, max_features=2000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} character n-grams")

# Fit Multinomial Naive Bayes
clf = MultinomialNB(alpha=1.0, fit_prior=True)
clf.fit(X_train_vec, y_train)

# Evaluate on Test Set
y_pred = clf.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("=" * 60)
print("  SCIKIT-LEARN MODEL EVALUATION RESULTS (TEST SET)")
print("=" * 60)
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall:    {rec * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")
print("\nConfusion Matrix:")
print(f"  True Negative (TN):  {cm[0, 0]:3d}  |  False Positive (FP): {cm[0, 1]:3d}")
print(f"  False Negative (FN): {cm[1, 0]:3d}  |  True Positive (TP):  {cm[1, 1]:3d}")
print("=" * 60)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['invalid', 'valid']))

## 4. Spot Checks on FLA-Lang Canonical Programs

In [ ]:
test_cases = [
    ("int x = 10;\nx = a + 5;", 1),
    ("result = (a + 5) * b - c / 2;", 1),
    ("if (x > 10) {\n  y = 1;\n} else {\n  y = 0;\n}", 1),
    ("while (count < 5) {\n  count = count + 1;\n}", 1),
    ("int x = 10", 0),
    ("x = (a + 5;", 0),
    ("int x = 10 @ 5;", 0),
    ("int 1x = 10;", 0)
]

print("Spot-check predictions on representative programs:")
for code, expected in test_cases:
    vec = vectorizer.transform([code])
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec)[0][pred]
    pred_label = "valid" if pred == 1 else "invalid"
    exp_label = "valid" if expected == 1 else "invalid"
    status = "✓ PASS" if pred == expected else "✗ FAIL"
    print(f"[{status}] Expected: {exp_label:7s} | Predicted: {pred_label:7s} ({prob*100:5.1f}%) | Code: {code.replace(chr(10), ' ')}")

## 5. Export Model Parameters to `ml_model.json`
Extracts the vocabulary mapping, class log priors, feature log probabilities, and evaluation metrics so the web application can run browser-side inference locally without any Python server.

In [ ]:
# Export exact parameters
export_payload = {
    "model_name": "FLA-Lang Code Validity Predictor",
    "algorithm": "Multinomial Naive Bayes",
    "feature_type": "Character N-Grams",
    "ngram_range": [1, 4],
    "classes": ["invalid", "valid"],
    "metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1_score": round(float(f1), 4),
        "confusion_matrix": {
            "tn": int(cm[0, 0]),
            "fp": int(cm[0, 1]),
            "fn": int(cm[1, 0]),
            "tp": int(cm[1, 1])
        },
        "train_samples": int(len(X_train)),
        "test_samples": int(len(X_test))
    },
    "class_log_priors": clf.class_log_prior_.tolist(),
    "vocabulary": {k: int(v) for k, v in vectorizer.vocabulary_.items()},
    "feature_log_prob": clf.feature_log_prob_.tolist()
}

with open("ml_model.json", "w", encoding="utf-8") as f:
    json.dump(export_payload, f, indent=2)

print(f"✓ ml_model.json successfully written ({len(json.dumps(export_payload))} bytes).")

## 6. Download Link for Google Colab

In [ ]:
# Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download('ml_model.json')
    print("Triggered download of ml_model.json to your computer.")
except ImportError:
    print("Running locally outside Colab. ml_model.json has been saved in the working directory.")

## 7. Instructions for Reproducing and Using in FLA-Lang
1. **Open Google Colab:** Visit [https://colab.research.google.com](https://colab.research.google.com) and click **Upload Notebook**, selecting `FLA_Lang_ML_Model.ipynb`.
2. **Run All Cells:** Click **Runtime > Run all** (or press `Ctrl+F9`).
3. **Download Model:** The final cell automatically triggers a browser download for `ml_model.json`.
4. **Deploy into Project:** Place the downloaded `ml_model.json` into the root directory of `FLA-Compiler-Frontend/`.
5. **Open Web App:** Open `index.html` in your web browser. The **ML Code Prediction** panel will automatically load the model and display live statistical validity predictions.